# Interpretabilidad

Un modelo que acierta pero no se puede explicar tiene poco valor en química médica: lo
que un químico necesita saber es **qué parte de la molécula** conviene cambiar. Este
capítulo abre el modelo.

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import Draw, rdFingerprintGenerator

datos = Path("../datos/procesados")
paquete = joblib.load("../modelos/modelo_final.joblib")
modelo, scaler, idx_te = paquete["modelo"], paquete["scaler"], paquete["idx_test"]

X_fp = np.load(datos / "X_fp.npy")
X_desc = pd.read_parquet(datos / "X_desc.parquet")
meta = pd.read_parquet(datos / "y_meta.parquet")

nombres = [f"bit_{i}" for i in range(X_fp.shape[1])] + list(X_desc.columns)
X = np.hstack([X_fp, scaler.transform(X_desc)]).astype(np.float32)
len(nombres)

## Importancia de variables

La importancia por impureza de los modelos de árbol está sesgada hacia variables con
muchos valores posibles. Para fingerprints binarios el sesgo es menor, pero la
**importancia por permutación** sigue siendo más fiable.

In [ ]:
from sklearn.inspection import permutation_importance

idx_tr = np.setdiff1d(np.arange(len(meta)), idx_te)
y = meta["pic50"].to_numpy()

# submuestra para que sea rápido
sub = np.random.default_rng(42).choice(idx_te, size=min(400, len(idx_te)), replace=False)

perm = permutation_importance(
    modelo, X[sub], y[sub], n_repeats=5, random_state=42, n_jobs=-1,
)

imp = pd.DataFrame({
    "variable": nombres,
    "importancia": perm.importances_mean,
    "sd": perm.importances_std,
}).nlargest(25, "importancia")

plt.figure(figsize=(8, 7))
plt.barh(imp["variable"][::-1], imp["importancia"][::-1], xerr=imp["sd"][::-1])
plt.xlabel("caída de R² al permutar")
plt.title("Importancia por permutación (top 25)")
plt.tight_layout()
imp.head(15)

## SHAP

Los valores de Shapley reparten la predicción entre las variables de forma consistente.
Para modelos de árbol, `TreeExplainer` los calcula de forma exacta y rápida.

In [ ]:
import shap

explainer = shap.TreeExplainer(modelo)
shap_vals = explainer.shap_values(X[sub])

shap.summary_plot(shap_vals, X[sub], feature_names=nombres, max_display=20, show=True)

En el *summary plot*:

- cada punto es un compuesto,
- el eje x es cuánto empuja esa variable la predicción (hacia más o menos potente),
- el color es el valor de la variable (para un bit: presente o ausente).

Un bit con puntos rojos a la derecha significa: *cuando este fragmento está presente, el
modelo predice más potencia*.

In [ ]:
# Dependencia de descriptores continuos relevantes
for var in ["MolLogP", "TPSA", "NumHDonors"]:
    if var in nombres:
        shap.dependence_plot(nombres.index(var), shap_vals, X[sub],
                             feature_names=nombres, show=True)

## De bits a fragmentos

Un bit de Morgan no dice nada por sí mismo; hay que recuperar la subestructura que lo
activa. Esto es lo que convierte el análisis en información química accionable.

In [ ]:
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def fragmentos_de_bit(bit, smiles_list, n=4):
    """Devuelve ejemplos del entorno molecular que activa un bit."""
    ejemplos = []
    for smi in smiles_list:
        m = Chem.MolFromSmiles(smi)
        if m is None:
            continue
        ao = rdFingerprintGenerator.AdditionalOutput()
        ao.AllocateBitInfoMap()
        gen.GetFingerprint(m, additionalOutput=ao)
        info = ao.GetBitInfoMap()
        if bit in info:
            atomo, radio = info[bit][0]
            ejemplos.append((m, bit, {bit: ((atomo, radio),)}))
        if len(ejemplos) >= n:
            break
    return ejemplos

# tomar los bits más importantes
bits_top = [int(v.split("_")[1]) for v in imp["variable"] if v.startswith("bit_")][:6]
smiles_all = meta["smiles"].tolist()

for b in bits_top[:3]:
    ej = fragmentos_de_bit(b, smiles_all)
    if ej:
        img = Draw.DrawMorganBits(ej, molsPerRow=4, legends=[f"bit {b}"] * len(ej))
        display(img)

## Explicación de una predicción concreta

Para un compuesto individual, el *waterfall plot* muestra qué variables suman y qué
variables restan respecto al valor base.

In [ ]:
i = 0  # índice dentro de la submuestra
shap.plots._waterfall.waterfall_legacy(
    explainer.expected_value,
    shap_vals[i],
    feature_names=nombres,
    max_display=15,
)
print(f"compuesto: {meta['chembl_id'].iloc[sub[i]]}")
print(f"observado: {y[sub[i]]:.2f}  predicho: {modelo.predict(X[sub[i]:sub[i]+1])[0]:.2f}")

In [ ]:
# Mapa de contribuciones sobre la estructura
from rdkit.Chem.Draw import SimilarityMaps

smi = meta["smiles"].iloc[sub[i]]
mol = Chem.MolFromSmiles(smi)

# contribución de cada átomo = suma de los SHAP de los bits que activa
contrib = np.zeros(mol.GetNumAtoms())
ao = rdFingerprintGenerator.AdditionalOutput()
ao.AllocateBitInfoMap()
gen.GetFingerprint(mol, additionalOutput=ao)

for bit, entornos in ao.GetBitInfoMap().items():
    if bit < X_fp.shape[1]:
        peso = shap_vals[i][bit]
        for atomo, _ in entornos:
            contrib[atomo] += peso

fig = SimilarityMaps.GetSimilarityMapFromWeights(mol, list(contrib), size=(350, 300))

```{warning}
SHAP explica **el modelo**, no la biología. Un fragmento con SHAP alto puede serlo porque
aparece en una serie congenérica muy potente del conjunto de entrenamiento, no porque
tenga un papel mecanístico. Toda hipótesis química derivada de aquí hay que contrastarla
con la literatura y, si es posible, con estructuras cristalográficas del complejo
quinasa–inhibidor (PDB).
```

## Contraste con conocimiento previo

Validación cualitativa: ¿coinciden los fragmentos destacados con farmacóforos conocidos
de inhibidores de quinasas?

- Aceptor/dador de hidrógeno que imita el ATP en la bisagra (residuos *hinge*).
- Núcleos de quinazolina, pirimidina, pirrolopirimidina — muy frecuentes en inhibidores
  de tipo I.
- Grupo amida o urea que alcanza el bolsillo trasero en inhibidores de tipo II.
- Acrilamida o cloroacetamida como *warhead* covalente.

Si el modelo destaca precisamente esos motivos, es una señal fuerte de que ha aprendido
química y no solo memoria de series.